# 🎨 AI Photo Generator — توليد الصور بالذكاء الاصطناعي

> **تأكد من تفعيل GPU:** Runtime → Change runtime type → T4 GPU

---
### خطوات التشغيل:
1. شغّل **Cell 1** لتثبيت المكتبات (مرة واحدة فقط)
2. ضع **ngrok token** في **Cell 2** ثم شغّله
3. انقر على الرابط الظاهر في الـ output

In [ ]:
# ============================================================
# Cell 1: تثبيت المكتبات  (شغّلها مرة واحدة فقط)
# ============================================================
!pip install -q fastapi uvicorn[standard] diffusers transformers accelerate \
    deep-translator pyngrok python-multipart Pillow xformers

# تثبيت PyTorch مع دعم CUDA (Colab يأتي به مثبتاً عادةً، لكن نضمن الإصدار الصحيح)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

print('✅ تم تثبيت جميع المكتبات بنجاح!')

In [ ]:
# ============================================================
# Cell 2: تشغيل التطبيق الكامل
# ============================================================
# ⚠️  ضع ngrok token الخاص بك هنا
#    احصل عليه مجاناً من: https://ngrok.com → Dashboard → Your Authtoken
NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"
# ============================================================

import os, io, base64, threading, warnings
warnings.filterwarnings('ignore')

import torch
from PIL import Image
from fastapi import FastAPI, Form
from fastapi.responses import HTMLResponse, JSONResponse
import uvicorn
from pyngrok import ngrok
from deep_translator import GoogleTranslator
from diffusers import StableDiffusionPipeline, AutoPipelineForText2Image

# ─────────────────────────────────────────────
# 1. تهيئة نموذج Stable Diffusion
# ─────────────────────────────────────────────
print('⏳ جارٍ تحميل نموذج Stable Diffusion...')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if device == 'cuda' else torch.float32

# نحاول SDXL-Turbo أولاً (سريع جداً)؛ إذا فشل نستخدم SD v1.5
pipe = None
model_name = ''
try:
    pipe = AutoPipelineForText2Image.from_pretrained(
        'stabilityai/sdxl-turbo',
        torch_dtype=dtype,
        variant='fp16'
    ).to(device)
    pipe.enable_attention_slicing()
    model_name = 'SDXL-Turbo'
    print(f'✅ تم تحميل {model_name} على {device}')
except Exception as e:
    print(f'⚠️  SDXL-Turbo غير متاح ({e})، جارٍ تحميل SD v1.5...')
    pipe = StableDiffusionPipeline.from_pretrained(
        'runwayml/stable-diffusion-v1-5',
        torch_dtype=dtype,
        safety_checker=None
    ).to(device)
    pipe.enable_attention_slicing()
    model_name = 'Stable Diffusion v1.5'
    print(f'✅ تم تحميل {model_name} على {device}')

# ─────────────────────────────────────────────
# 2. وظيفة الترجمة
# ─────────────────────────────────────────────
def translate_to_english(text: str) -> tuple[str, bool]:
    """ترجم النص إلى الإنجليزية إذا كان يحتوي على عربية.
    
    Returns:
        (translated_text, was_translated)
    """
    arabic_chars = sum(1 for ch in text if '\u0600' <= ch <= '\u06FF')
    if arabic_chars / max(len(text), 1) > 0.2:
        try:
            translated = GoogleTranslator(source='ar', target='en').translate(text)
            return translated, True
        except Exception:
            pass
    return text, False

# ─────────────────────────────────────────────
# 3. HTML / CSS / JS الواجهة الأمامية
# ─────────────────────────────────────────────
HTML_PAGE = """
<!DOCTYPE html>
<html lang="ar" dir="rtl">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>🎨 AI Photo Generator</title>

  <!-- Google Fonts -->
  <link href="https://fonts.googleapis.com/css2?family=Tajawal:wght@300;400;500;700;900&family=Inter:wght@300;400;600;700&display=swap" rel="stylesheet" />

  <!-- Tailwind CSS -->
  <script src="https://cdn.tailwindcss.com"></script>
  <script>
    tailwind.config = {
      theme: {
        extend: {
          colors: {
            purple: {
              400: '#c084fc',
              500: '#a855f7',
              600: '#9333ea',
              700: '#7c3aed',
              800: '#6d28d9',
              900: '#4c1d95',
            }
          },
          fontFamily: {
            tajawal: ['Tajawal', 'sans-serif'],
            inter: ['Inter', 'sans-serif'],
          },
          animation: {
            'pulse-glow': 'pulse-glow 2s ease-in-out infinite',
            'fade-in': 'fade-in 0.6s ease-out forwards',
            'spin-slow': 'spin 1.5s linear infinite',
          },
          keyframes: {
            'pulse-glow': {
              '0%, 100%': { boxShadow: '0 0 20px rgba(168,85,247,0.4)' },
              '50%':       { boxShadow: '0 0 40px rgba(168,85,247,0.8)' },
            },
            'fade-in': {
              '0%':   { opacity: '0', transform: 'translateY(20px)' },
              '100%': { opacity: '1', transform: 'translateY(0)' },
            }
          }
        }
      }
    };
  </script>

  <!-- Lucide Icons -->
  <script src="https://unpkg.com/lucide@latest/dist/umd/lucide.min.js"></script>

  <style>
    * { box-sizing: border-box; margin: 0; padding: 0; }

    body {
      background-color: #000;
      font-family: 'Tajawal', 'Inter', sans-serif;
      color: #f0f0f0;
      min-height: 100vh;
      overflow-x: hidden;
    }

    /* Animated background grid */
    body::before {
      content: '';
      position: fixed;
      inset: 0;
      background-image:
        linear-gradient(rgba(124,58,237,0.04) 1px, transparent 1px),
        linear-gradient(90deg, rgba(124,58,237,0.04) 1px, transparent 1px);
      background-size: 40px 40px;
      pointer-events: none;
      z-index: 0;
    }

    /* Glassmorphism cards */
    .glass {
      background: rgba(255,255,255,0.04);
      backdrop-filter: blur(16px);
      -webkit-backdrop-filter: blur(16px);
      border: 1px solid rgba(168,85,247,0.15);
      border-radius: 16px;
    }

    .glass-strong {
      background: rgba(255,255,255,0.07);
      backdrop-filter: blur(24px);
      -webkit-backdrop-filter: blur(24px);
      border: 1px solid rgba(168,85,247,0.25);
      border-radius: 20px;
    }

    /* Purple glow button */
    .btn-glow {
      background: linear-gradient(135deg, #7c3aed, #a855f7);
      border: none;
      border-radius: 12px;
      color: #fff;
      cursor: pointer;
      font-family: 'Tajawal', sans-serif;
      font-size: 1.05rem;
      font-weight: 700;
      padding: 14px 32px;
      transition: all 0.3s ease;
      box-shadow: 0 0 20px rgba(168,85,247,0.4);
      position: relative;
      overflow: hidden;
    }
    .btn-glow:hover:not(:disabled) {
      transform: translateY(-2px);
      box-shadow: 0 0 40px rgba(168,85,247,0.7);
    }
    .btn-glow:disabled {
      opacity: 0.6;
      cursor: not-allowed;
      transform: none;
    }
    .btn-glow::before {
      content: '';
      position: absolute;
      top: -50%;
      left: -60%;
      width: 40%;
      height: 200%;
      background: rgba(255,255,255,0.15);
      transform: skewX(-20deg);
      transition: left 0.5s ease;
    }
    .btn-glow:hover::before { left: 120%; }

    /* Download button */
    .btn-outline {
      background: transparent;
      border: 1.5px solid rgba(168,85,247,0.6);
      border-radius: 10px;
      color: #c084fc;
      cursor: pointer;
      font-family: 'Tajawal', sans-serif;
      font-size: 0.95rem;
      font-weight: 600;
      padding: 10px 24px;
      transition: all 0.3s ease;
    }
    .btn-outline:hover {
      background: rgba(168,85,247,0.15);
      box-shadow: 0 0 20px rgba(168,85,247,0.3);
      color: #e9d5ff;
    }

    /* Range slider */
    input[type=range] {
      -webkit-appearance: none;
      width: 100%;
      height: 5px;
      border-radius: 3px;
      background: rgba(168,85,247,0.2);
      outline: none;
    }
    input[type=range]::-webkit-slider-thumb {
      -webkit-appearance: none;
      width: 18px;
      height: 18px;
      border-radius: 50%;
      background: linear-gradient(135deg, #7c3aed, #a855f7);
      cursor: pointer;
      box-shadow: 0 0 8px rgba(168,85,247,0.6);
    }

    /* Textarea */
    .prompt-input {
      background: rgba(255,255,255,0.05);
      border: 1.5px solid rgba(168,85,247,0.2);
      border-radius: 14px;
      color: #f0f0f0;
      font-family: 'Tajawal', sans-serif;
      font-size: 1rem;
      line-height: 1.6;
      padding: 16px;
      resize: none;
      transition: border-color 0.3s, box-shadow 0.3s;
      width: 100%;
    }
    .prompt-input::placeholder { color: rgba(255,255,255,0.3); }
    .prompt-input:focus {
      border-color: rgba(168,85,247,0.6);
      box-shadow: 0 0 20px rgba(168,85,247,0.2);
      outline: none;
    }

    /* Status badge */
    .status-dot {
      width: 9px; height: 9px;
      border-radius: 50%;
      display: inline-block;
      animation: blink 1.5s ease-in-out infinite;
    }
    @keyframes blink {
      0%,100% { opacity: 1; }
      50%      { opacity: 0.3; }
    }

    /* Spinner */
    .spinner {
      width: 22px; height: 22px;
      border: 3px solid rgba(255,255,255,0.3);
      border-top-color: #fff;
      border-radius: 50%;
      animation: spin 0.8s linear infinite;
    }
    @keyframes spin { to { transform: rotate(360deg); } }

    /* Image fade-in */
    @keyframes fade-in {
      from { opacity: 0; transform: scale(0.96) translateY(10px); }
      to   { opacity: 1; transform: scale(1)   translateY(0); }
    }
    .img-reveal { animation: fade-in 0.7s ease-out forwards; }

    /* Scrollbar */
    ::-webkit-scrollbar { width: 6px; }
    ::-webkit-scrollbar-track { background: transparent; }
    ::-webkit-scrollbar-thumb { background: rgba(168,85,247,0.4); border-radius: 3px; }

    /* Select */
    select {
      background: rgba(255,255,255,0.07);
      border: 1px solid rgba(168,85,247,0.25);
      border-radius: 8px;
      color: #f0f0f0;
      font-family: 'Tajawal', sans-serif;
      font-size: 0.9rem;
      padding: 8px 12px;
      width: 100%;
      cursor: pointer;
    }
    select option { background: #1a0a2e; }
    select:focus { border-color: rgba(168,85,247,0.6); outline: none; }

    /* Negative prompt */
    .neg-input {
      background: rgba(255,255,255,0.03);
      border: 1px solid rgba(255,60,60,0.2);
      border-radius: 10px;
      color: #f0f0f0;
      font-family: 'Tajawal', sans-serif;
      font-size: 0.88rem;
      padding: 10px 14px;
      resize: none;
      width: 100%;
    }
    .neg-input::placeholder { color: rgba(255,100,100,0.4); }
    .neg-input:focus { border-color: rgba(255,80,80,0.5); outline: none; }
  </style>
</head>

<body class="relative">

  <!-- ░░ HEADER ░░ -->
  <header class="relative z-10 flex items-center justify-between px-8 py-5 border-b border-purple-900/30">
    <div class="flex items-center gap-3">
      <!-- Logo icon -->
      <div class="w-10 h-10 rounded-xl flex items-center justify-center"
           style="background:linear-gradient(135deg,#7c3aed,#a855f7);box-shadow:0 0 20px rgba(168,85,247,.5)">
        <i data-lucide="sparkles" class="w-5 h-5 text-white"></i>
      </div>
      <div>
        <h1 class="text-xl font-bold leading-tight"
            style="background:linear-gradient(90deg,#c084fc,#e879f9);-webkit-background-clip:text;-webkit-text-fill-color:transparent">
          AI Photo Generator
        </h1>
        <p class="text-xs text-purple-400/70 font-light">توليد الصور بالذكاء الاصطناعي</p>
      </div>
    </div>

    <!-- Model badge -->
    <div id="modelBadge" class="glass flex items-center gap-2 px-4 py-2 text-sm">
      <span class="status-dot" style="background:#a855f7"></span>
      <span id="modelName" class="text-purple-300 font-medium">MODEL_NAME</span>
    </div>
  </header>

  <!-- ░░ LAYOUT ░░ -->
  <div class="relative z-10 flex gap-6 p-6" style="min-height:calc(100vh - 80px)">

    <!-- ─── SIDEBAR ─── -->
    <aside class="glass-strong flex-shrink-0 p-6 flex flex-col gap-6" style="width:280px">

      <h2 class="text-sm font-bold text-purple-300 uppercase tracking-widest flex items-center gap-2">
        <i data-lucide="settings-2" class="w-4 h-4"></i> الإعدادات
      </h2>

      <!-- Image Size -->
      <div>
        <label class="block text-xs text-gray-400 mb-2 flex items-center gap-1">
          <i data-lucide="frame" class="w-3 h-3"></i> أبعاد الصورة
        </label>
        <select id="imgSize">
          <option value="512,512">512 × 512  (سريع)</option>
          <option value="640,640">640 × 640</option>
          <option value="768,512">768 × 512  (أفقي)</option>
          <option value="512,768">512 × 768  (عمودي)</option>
          <option value="768,768" selected>768 × 768  (متوازن)</option>
          <option value="1024,1024">1024 × 1024  (عالي الدقة)</option>
        </select>
      </div>

      <!-- Steps slider -->
      <div>
        <label class="block text-xs text-gray-400 mb-2 flex items-center justify-between">
          <span class="flex items-center gap-1">
            <i data-lucide="sliders-horizontal" class="w-3 h-3"></i> خطوات الاستدلال
          </span>
          <span id="stepsVal" class="text-purple-400 font-bold">20</span>
        </label>
        <input type="range" id="stepsSlider" min="1" max="50" value="20"
               oninput="document.getElementById('stepsVal').textContent=this.value" />
        <div class="flex justify-between text-xs text-gray-600 mt-1">
          <span>1 (سريع جداً)</span><span>50 (دقيق جداً)</span>
        </div>
      </div>

      <!-- Guidance scale -->
      <div>
        <label class="block text-xs text-gray-400 mb-2 flex items-center justify-between">
          <span class="flex items-center gap-1">
            <i data-lucide="target" class="w-3 h-3"></i> قوة الالتزام بالنص
          </span>
          <span id="cfgVal" class="text-purple-400 font-bold">7.5</span>
        </label>
        <input type="range" id="cfgSlider" min="1" max="20" step="0.5" value="7.5"
               oninput="document.getElementById('cfgVal').textContent=parseFloat(this.value).toFixed(1)" />
        <div class="flex justify-between text-xs text-gray-600 mt-1">
          <span>1 (إبداعي)</span><span>20 (حرفي)</span>
        </div>
      </div>

      <!-- Negative prompt -->
      <div>
        <label class="block text-xs text-gray-400 mb-2 flex items-center gap-1">
          <i data-lucide="ban" class="w-3 h-3 text-red-400"></i>
          <span>Negative Prompt</span>
          <span class="text-gray-600">(ما تريد إزالته)</span>
        </label>
        <textarea id="negPrompt" class="neg-input" rows="3"
          placeholder="blurry, low quality, distorted, ugly..."></textarea>
      </div>

      <!-- Seed -->
      <div>
        <label class="block text-xs text-gray-400 mb-2 flex items-center gap-1">
          <i data-lucide="shuffle" class="w-3 h-3"></i> Seed
          <span class="text-gray-600">(فارغ = عشوائي)</span>
        </label>
        <input type="number" id="seedInput"
               class="w-full bg-white/5 border border-purple-900/30 rounded-lg px-3 py-2 text-sm text-gray-300 focus:border-purple-500 focus:outline-none"
               placeholder="مثال: 42" />
      </div>

      <!-- Divider -->
      <div class="border-t border-purple-900/20 pt-4">
        <div class="flex items-center gap-2 text-xs text-gray-500">
          <i data-lucide="cpu" class="w-3 h-3 text-purple-400"></i>
          <span id="deviceInfo">جارٍ الفحص...</span>
        </div>
      </div>
    </aside>

    <!-- ─── MAIN AREA ─── -->
    <main class="flex-1 flex flex-col gap-6">

      <!-- Prompt input card -->
      <div class="glass-strong p-6">
        <label class="block text-sm font-semibold text-purple-300 mb-3 flex items-center gap-2">
          <i data-lucide="pencil-line" class="w-4 h-4"></i>
          وصف الصورة (بالعربية أو الإنجليزية)
        </label>

        <!-- Arabic detection banner -->
        <div id="arabicBanner" class="hidden mb-3 flex items-center gap-2 text-xs px-3 py-2 rounded-lg"
             style="background:rgba(168,85,247,0.1);border:1px solid rgba(168,85,247,0.3)">
          <i data-lucide="languages" class="w-3 h-3 text-purple-400"></i>
          <span class="text-purple-300">تم اكتشاف اللغة العربية — سيتم الترجمة تلقائياً إلى الإنجليزية قبل التوليد</span>
        </div>

        <textarea id="promptInput" class="prompt-input" rows="4"
          placeholder="صف الصورة التي تريد توليدها... مثال: غابة سحرية في الليل مضيئة بضوء القمر"></textarea>

        <!-- Action row -->
        <div class="flex items-center justify-between mt-4 flex-wrap gap-3">
          <div class="flex items-center gap-3">
            <button class="btn-glow flex items-center gap-2" id="generateBtn" onclick="generateImage()">
              <i data-lucide="wand-2" class="w-4 h-4" id="wand"></i>
              <span id="btnText">✨ توليد الصورة</span>
              <div class="spinner hidden" id="spinner"></div>
            </button>
            <button class="btn-outline flex items-center gap-1" onclick="clearAll()">
              <i data-lucide="refresh-ccw" class="w-3 h-3"></i> مسح
            </button>
          </div>
          <div id="translatedBadge" class="hidden text-xs px-3 py-1 rounded-full font-medium"
               style="background:rgba(168,85,247,0.15);border:1px solid rgba(168,85,247,0.4);color:#c084fc">
            <i data-lucide="check-circle" class="w-3 h-3 inline mr-1"></i>
            <span id="translatedText"></span>
          </div>
        </div>
      </div>

      <!-- Error banner -->
      <div id="errorBanner" class="hidden glass p-4 flex items-start gap-3"
           style="border-color:rgba(239,68,68,0.3)">
        <i data-lucide="alert-triangle" class="w-5 h-5 text-red-400 flex-shrink-0 mt-0.5"></i>
        <div>
          <p class="text-sm font-semibold text-red-400">حدث خطأ</p>
          <p id="errorMsg" class="text-xs text-red-300 mt-1"></p>
        </div>
      </div>

      <!-- Image result card -->
      <div class="glass-strong flex-1 p-6 flex flex-col">
        <div class="flex items-center justify-between mb-4">
          <span class="text-sm font-semibold text-purple-300 flex items-center gap-2">
            <i data-lucide="image" class="w-4 h-4"></i> الصورة الناتجة
          </span>
          <div id="downloadArea" class="hidden flex items-center gap-2">
            <span id="genInfo" class="text-xs text-gray-500"></span>
            <button class="btn-outline flex items-center gap-1" id="downloadBtn" onclick="downloadImage()">
              <i data-lucide="download" class="w-3 h-3"></i> تحميل الصورة
            </button>
          </div>
        </div>

        <!-- Placeholder -->
        <div id="placeholder" class="flex-1 flex flex-col items-center justify-center py-16"
             style="border:2px dashed rgba(168,85,247,0.15);border-radius:12px">
          <div class="w-20 h-20 rounded-full flex items-center justify-center mb-4"
               style="background:rgba(168,85,247,0.08);border:1px solid rgba(168,85,247,0.2)">
            <i data-lucide="image-plus" class="w-9 h-9 text-purple-800"></i>
          </div>
          <p class="text-gray-600 text-sm">ستظهر الصورة هنا بعد التوليد</p>
          <p class="text-gray-700 text-xs mt-1">اكتب وصفاً واضغط على زر التوليد</p>
        </div>

        <!-- Loading animation -->
        <div id="loadingArea" class="hidden flex-1 flex flex-col items-center justify-center py-16">
          <div class="relative w-32 h-32 mb-6">
            <div class="absolute inset-0 rounded-full"
                 style="border:3px solid rgba(168,85,247,0.2)"></div>
            <div class="absolute inset-0 rounded-full animate-spin"
                 style="border:3px solid transparent;border-top-color:#a855f7"></div>
            <div class="absolute inset-4 rounded-full"
                 style="border:2px solid rgba(168,85,247,0.15)"></div>
            <div class="absolute inset-4 rounded-full"
                 style="border:2px solid transparent;border-top-color:#7c3aed;animation:spin 2s linear infinite reverse"></div>
            <div class="absolute inset-0 flex items-center justify-center">
              <i data-lucide="sparkles" class="w-8 h-8 text-purple-500"></i>
            </div>
          </div>
          <p class="text-purple-300 font-medium">جارٍ توليد الصورة...</p>
          <p id="loadingStep" class="text-gray-600 text-sm mt-1">يرجى الانتظار، قد يستغرق هذا 10–60 ثانية</p>
          <div class="mt-4 w-64 h-1.5 rounded-full overflow-hidden" style="background:rgba(168,85,247,0.1)">
            <div id="progressBar" class="h-full rounded-full"
                 style="background:linear-gradient(90deg,#7c3aed,#a855f7);width:0%;transition:width 0.5s ease"></div>
          </div>
        </div>

        <!-- Generated image -->
        <div id="imageArea" class="hidden flex-1 flex flex-col items-center justify-center">
          <img id="resultImg" src="" alt="Generated Image"
               class="img-reveal max-w-full rounded-2xl"
               style="max-height:70vh;object-fit:contain;box-shadow:0 0 40px rgba(168,85,247,0.3);" />
        </div>
      </div>
    </main>
  </div>

  <!-- ░░ JAVASCRIPT ░░ -->
  <script>
    // Init Lucide icons
    lucide.createIcons();

    // ── Arabic detection ──
    const promptEl    = document.getElementById('promptInput');
    const arabicBanner = document.getElementById('arabicBanner');

    promptEl.addEventListener('input', () => {
      const txt = promptEl.value;
      const arabicCount = (txt.match(/[\u0600-\u06FF]/g) || []).length;
      const isArabic = arabicCount / Math.max(txt.length, 1) > 0.2;
      arabicBanner.classList.toggle('hidden', !isArabic);
      // Auto-direction
      promptEl.style.direction = isArabic ? 'rtl' : 'ltr';
      lucide.createIcons();
    });

    // ── Progress bar simulation ──
    let progressInterval = null;

    function startProgress() {
      let p = 0;
      const bar = document.getElementById('progressBar');
      const steps = [
        [0,  15, 800,  'جارٍ معالجة النص...'],
        [15, 40, 2000, 'توليد النويز الأولي...'],
        [40, 75, 5000, 'معالجة الخطوات الاستدلالية...'],
        [75, 90, 3000, 'تحسين جودة الصورة...'],
      ];
      let si = 0;
      const stepEl = document.getElementById('loadingStep');

      function tick() {
        if (si >= steps.length) return;
        const [from, to, dur, label] = steps[si];
        stepEl.textContent = label;
        const inc = (to - from) / (dur / 50);
        progressInterval = setInterval(() => {
          p = Math.min(p + inc, to);
          bar.style.width = p + '%';
          if (p >= to) {
            clearInterval(progressInterval);
            si++;
            tick();
          }
        }, 50);
      }
      tick();
    }

    function stopProgress() {
      clearInterval(progressInterval);
      document.getElementById('progressBar').style.width = '100%';
    }

    // ── Show / hide sections ──
    function showSection(id) {
      ['placeholder','loadingArea','imageArea'].forEach(s =>
        document.getElementById(s).classList.add('hidden')
      );
      document.getElementById(id).classList.remove('hidden');
    }

    // ── Generate ──
    async function generateImage() {
      const prompt = promptEl.value.trim();
      if (!prompt) {
        promptEl.focus();
        promptEl.style.borderColor = 'rgba(239,68,68,0.6)';
        setTimeout(() => { promptEl.style.borderColor = ''; }, 1500);
        return;
      }

      const [w, h]    = document.getElementById('imgSize').value.split(',').map(Number);
      const steps     = parseInt(document.getElementById('stepsSlider').value);
      const cfg       = parseFloat(document.getElementById('cfgSlider').value);
      const negPrompt = document.getElementById('negPrompt').value.trim();
      const seedRaw   = document.getElementById('seedInput').value.trim();
      const seed      = seedRaw ? parseInt(seedRaw) : -1;

      // UI: loading state
      const btn    = document.getElementById('generateBtn');
      const wand   = document.getElementById('wand');
      const sp     = document.getElementById('spinner');
      const btnTxt = document.getElementById('btnText');
      btn.disabled = true;
      wand.classList.add('hidden');
      sp.classList.remove('hidden');
      btnTxt.textContent = 'جارٍ التوليد...';

      document.getElementById('errorBanner').classList.add('hidden');
      document.getElementById('translatedBadge').classList.add('hidden');
      document.getElementById('downloadArea').classList.add('hidden');
      showSection('loadingArea');
      startProgress();

      try {
        const fd = new FormData();
        fd.append('prompt', prompt);
        fd.append('width',  w);
        fd.append('height', h);
        fd.append('steps',  steps);
        fd.append('guidance_scale', cfg);
        fd.append('negative_prompt', negPrompt);
        fd.append('seed', seed);

        const res  = await fetch('/generate', { method: 'POST', body: fd });
        const data = await res.json();

        if (!res.ok || data.error) {
          throw new Error(data.error || `HTTP ${res.status}`);
        }

        stopProgress();

        // Show translation badge
        if (data.translated) {
          const badge  = document.getElementById('translatedBadge');
          const txtEl  = document.getElementById('translatedText');
          txtEl.textContent = `تمت الترجمة: "${data.english_prompt}"`.substring(0, 80);
          badge.classList.remove('hidden');
        }

        // Show image
        const img = document.getElementById('resultImg');
        img.src = 'data:image/png;base64,' + data.image;
        img.classList.remove('img-reveal');
        void img.offsetWidth; // reflow for re-animation
        img.classList.add('img-reveal');
        showSection('imageArea');

        // Show download & info
        document.getElementById('genInfo').textContent =
          `${w}×${h}px · ${steps} خطوة · ${data.time_sec}ث`;
        document.getElementById('downloadArea').classList.remove('hidden');

        lucide.createIcons();

      } catch (err) {
        stopProgress();
        showSection('placeholder');
        const eb = document.getElementById('errorBanner');
        document.getElementById('errorMsg').textContent = err.message;
        eb.classList.remove('hidden');
        lucide.createIcons();
      } finally {
        btn.disabled = false;
        wand.classList.remove('hidden');
        sp.classList.add('hidden');
        btnTxt.textContent = '✨ توليد الصورة';
      }
    }

    // ── Download ──
    function downloadImage() {
      const img = document.getElementById('resultImg');
      if (!img.src || img.src === window.location.href) return;
      const a = document.createElement('a');
      a.href = img.src;
      a.download = 'ai-generated-' + Date.now() + '.png';
      a.click();
    }

    // ── Clear all ──
    function clearAll() {
      promptEl.value = '';
      document.getElementById('negPrompt').value = '';
      arabicBanner.classList.add('hidden');
      document.getElementById('translatedBadge').classList.add('hidden');
      document.getElementById('errorBanner').classList.add('hidden');
      document.getElementById('downloadArea').classList.add('hidden');
      showSection('placeholder');
    }

    // ── Device info (fetched from backend) ──
    fetch('/status').then(r => r.json()).then(d => {
      document.getElementById('deviceInfo').textContent = d.device_info || 'غير معروف';
      document.getElementById('modelName').textContent  = d.model_name  || 'نموذج AI';
    }).catch(() => {
      document.getElementById('deviceInfo').textContent = 'غير متصل';
    });

    // Enter key support
    promptEl.addEventListener('keydown', e => {
      if (e.key === 'Enter' && e.ctrlKey) generateImage();
    });
  </script>
</body>
</html>
"""

# ─────────────────────────────────────────────
# 4. FastAPI Application
# ─────────────────────────────────────────────
app = FastAPI(title='AI Photo Generator')

@app.get('/', response_class=HTMLResponse)
async def root():
    return HTML_PAGE

@app.get('/status')
async def status():
    vram = ''
    if torch.cuda.is_available():
        vram_mb = torch.cuda.get_device_properties(0).total_memory // (1024**2)
        vram = f' | VRAM: {vram_mb} MB'
    device_info = f"{'GPU ✅' if torch.cuda.is_available() else 'CPU ⚠️'}{vram}"
    return {'device_info': device_info, 'model_name': model_name}

@app.post('/generate')
async def generate(
    prompt:          str   = Form(...),
    width:           int   = Form(768),
    height:          int   = Form(768),
    steps:           int   = Form(20),
    guidance_scale:  float = Form(7.5),
    negative_prompt: str   = Form(''),
    seed:            int   = Form(-1),
):
    import time
    try:
        # ── Translate if Arabic ──
        english_prompt, was_translated = translate_to_english(prompt)

        # ── Clamp dimensions to multiples of 8 ──
        width  = max(256, min(1024, (width  // 8) * 8))
        height = max(256, min(1024, (height // 8) * 8))
        steps  = max(1,  min(50, steps))

        # ── Seed ──
        generator = None
        if seed != -1:
            generator = torch.Generator(device=device).manual_seed(seed)

        # ── Generate ──
        t0 = time.time()

        gen_kwargs = dict(
            prompt=english_prompt,
            width=width,
            height=height,
            num_inference_steps=steps,
            generator=generator,
        )

        # SDXL-Turbo uses guidance_scale=0 by default; SD v1.5 uses ~7.5
        if model_name == 'SDXL-Turbo':
            gen_kwargs['guidance_scale'] = 0.0
            gen_kwargs['num_inference_steps'] = min(steps, 4)  # turbo is 1-4 steps
        else:
            gen_kwargs['guidance_scale'] = guidance_scale
            if negative_prompt:
                gen_kwargs['negative_prompt'] = negative_prompt

        result = pipe(**gen_kwargs)
        elapsed = round(time.time() - t0, 1)

        image: Image.Image = result.images[0]

        # ── Encode to Base64 ──
        buf = io.BytesIO()
        image.save(buf, format='PNG', optimize=True)
        img_b64 = base64.b64encode(buf.getvalue()).decode('utf-8')

        return JSONResponse({
            'image':          img_b64,
            'translated':     was_translated,
            'english_prompt': english_prompt,
            'time_sec':       elapsed,
            'model':          model_name,
            'width':          width,
            'height':         height,
        })

    except Exception as exc:
        import traceback
        return JSONResponse({'error': str(exc), 'traceback': traceback.format_exc()}, status_code=500)

# ─────────────────────────────────────────────
# 5. تشغيل الخادم + ngrok
# ─────────────────────────────────────────────
if not NGROK_TOKEN or NGROK_TOKEN == 'YOUR_NGROK_TOKEN_HERE':
    print('⚠️  لم يتم تعيين ngrok token!')
    print('   احصل على توكنك المجاني من: https://ngrok.com → Dashboard → Your Authtoken')
    print('   ثم ضعه في المتغير NGROK_TOKEN في أعلى هذه الخلية')
else:
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(8000)
    print(f'\n🌐 ═══════════════════════════════════════')
    print(f'   رابط التطبيق العام: {public_url}')
    print(f'   افتحه من هاتفك أو متصفحك!')
    print(f'═══════════════════════════════════════\n')

    # تشغيل FastAPI في thread منفصل حتى لا يحجب Colab
    def run_server():
        uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

    thread = threading.Thread(target=run_server, daemon=True)
    thread.start()
    print('✅ الخادم يعمل الآن على المنفذ 8000')
    print('   (اتركه يعمل — لا تضغط على إيقاف)')